In [ ]:
import pickle as pkl

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
#读取训练数据
trainData = pd.read_csv(filepath_or_buffer='trainData.csv')
#读取测试数据
testData = pd.read_csv(filepath_or_buffer='testData.csv')
with open(file='batches.meta', mode='rb') as f:
    #读取标签数据
    labels = pkl.load(file=f)
label_names = labels['label_names']
trainData.shape, testData.shape, label_names

In [ ]:
#将数据存为二进制文件
with open(file='dataBatches', mode='wb') as f:
    pkl.dump((trainData, testData, label_names), f)

In [ ]:
#读取数据
with open(file='dataBatches', mode='rb') as f:
    trainData, testData, label_names = pkl.load(file=f)

In [ ]:
#获取训练数据图片，去掉最后一列
images = trainData.iloc[:, :-1]
#获取标签值，取最后一列
labels = trainData.iloc[:, -1]
#获取标签数量
categories_num = len(label_names)
for i, c in enumerate(label_names):
    #从每个类别中随机选择7张图片，无放回
    image_list = images[labels == i].sample(7, replace=False)
    for j, ind in enumerate(image_list.index):
        plt.subplot(7, categories_num, j * categories_num + i + 1)
        #先将单独的图片还原为3*32*32的原始形状
        #再通过transpose函数将通道纬度移到最后，修改为32*32*3的形状
        img = image_list.loc[ind].values.reshape(3, 32, 32).transpose(1, 2, 0)
        #显示图片
        plt.imshow(img)
        #关闭做标轴
        plt.axis('off')
        if j == 0:
            plt.title(label=c, fontdict={
                'fontsize': 6
            })
plt.show()


In [ ]:
import torch

torch.cuda.is_available()


#重新采样n条数据集
def get_samples(data, n=200):
    return data.groupby('label').sample(n, replace=False)


sample_data = get_samples(trainData, n=2000)
sample_data.shape


In [ ]:
#搭建全连接神经网络
import torch.nn as nn  #导入网络神经模块
import torch.nn.functional as F  #神经模块的函数模版
from torch.utils.data import TensorDataset, DataLoader


class FCNet(nn.Module):
    def __init__(self):
        super(FCNet, self).__init__()
        self.fc1 = nn.Linear(in_features=3 * 32 * 32, out_features=64)  #定义第一层全连接
        self.fc2 = nn.Linear(in_features=64, out_features=128)
        self.fc3 = nn.Linear(in_features=128, out_features=10)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        x = F.softmax(input=x, dim=1)
        return x


model1 = FCNet()

#打包数据集
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(sample_data, test_size=0.2, random_state=42)
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_val = val_data.iloc[:, :-1]
y_val = val_data.iloc[:, -1]

#特征和标签绑定
train_dataset = TensorDataset(
    torch.tensor(data=X_train.values, dtype=torch.float32),
    torch.tensor(data=y_train.values, dtype=torch.long)
)

val_dataset = TensorDataset(
    torch.tensor(data=X_val.values, dtype=torch.float32),
    torch.tensor(data=y_val.values, dtype=torch.long)
)

#创建DataLoader对象，设置批量大小和是否打乱数据
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=64, shuffle=False)

In [ ]:
import torch.optim as optim  #导入优化器模块


def train(model, train_loader, lr=0.01, num_epoches=10):
    model.train()  # 设置模型为训练模式,训练模式会启用梯度计算和参数更新
    optimizer = optim.SGD(model.parameters(), lr=lr)  # 创建Adam优化器对象，传入模型的参数和学习率
    criterion = nn.CrossEntropyLoss()  # 定义损失函数为均方误差损失函数
    for epoch in range(num_epoches):  # 循环训练指定的轮数
        total_loss = 0  # 初始化总损失为0
        for X_batch, y_batch in train_loader:  # 遍历训练数据加载器中的每个批次数据
            optimizer.zero_grad()  # 清零优化器的梯度缓存
            outputs = model(X_batch)  # 将输入数据通过模型进行前向传播，得到输出结果
            loss = criterion(outputs, y_batch)  # 计算损失函数，使用one-hot编码将标签转换为与输出维度相同的格式
            loss.backward()  # 反向传播计算梯度
            optimizer.step()  # 更新模型参数
            total_loss += loss.item()  # 累加当前批次的损失值

        avg_loss = total_loss / len(train_loader)  # 计算平均损失值
        print(f"Epoch {epoch + 1}/{num_epoches}, Loss: {avg_loss:.4f}")  # 打印当前轮数和平均损失值


def evaluate(model, val_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch)
            _, y_pred = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (y_pred.eq(y_batch).cpu().sum())
    accuracy = correct / total
    print(f'Validation Accuracy: {accuracy:.4f}')


train(model1, train_loader)
evaluate(model1, val_loader)


